# 🚀 Custom AI Enhancer — Kaggle Cloud GPU Automated Training Pipeline
- Model: CodeFormer Stage III CFT Fine-Tuning with ArcFace Identity Loss
- Hardware: Kaggle Nvidia T4 / P100 GPU
- Target: 20,000 Iterations & Static INT8 ONNX Export

In [ ]:
import os, sys, subprocess, shutil, torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU!")

In [ ]:
# 1. Clone repository
%cd /kaggle/working
if os.path.exists('custom-ai-enhancer'):
    shutil.rmtree('custom-ai-enhancer')
!git clone https://github.com/supli6669/Enhance-Image.git custom-ai-enhancer
%cd /kaggle/working/custom-ai-enhancer

# 2. Install dependencies
!pip install -q facexlib lpips gdown onnx onnxruntime-gpu pyyaml opencv-python scikit-image
!python tools/patch_and_install_basicsr.py

# 3. Download weights
!python tools/download_weights.py
print("Environment and weights ready!")

In [ ]:
# 4. Launch GPU Training with ArcFace Identity Loss
%cd /kaggle/working/custom-ai-enhancer
!python train_custom.py

In [ ]:
# 5. Export to ONNX & INT8 Quantization and copy to /kaggle/working for download
%cd /kaggle/working/custom-ai-enhancer
!python tools/export_onnx.py --model codeformer --input-pth weights/CodeFormer/codeformer.pth --output-onnx /kaggle/working/codeformer_v3.onnx
!python tools/quantize_onnx_static.py
if os.path.exists('weights/CodeFormer/codeformer_int8_v2.onnx'):
    shutil.copy('weights/CodeFormer/codeformer_int8_v2.onnx', '/kaggle/working/codeformer_int8_v3.onnx')
print("Training and Export finished! Outputs saved in /kaggle/working/")